# 04 — Stitching, Cumulative and Hydro-Mechanical Analysis

## Reproducible Groundwater–InSAR Workflow

This notebook reconstructs the previously completed terminal analyses:

1. **Stage 11 — SBAS stack-boundary QC**
2. **Stage 12 — Stitching of cumulative SBAS stacks**
3. **Stage 13 — Cumulative groundwater–InSAR monthly analysis**
4. **Stage 14 — Monthly hydro-mechanical states**

The purpose of this notebook is reproducibility and documentation of the historical workflow.  
No new scientific model is introduced here.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from groundwater_insar.config import load_config
from groundwater_insar.stage11_stitching_qc import reconstruct_stage11
from groundwater_insar.stage12_stitched_cumulative import reconstruct_stage12
from groundwater_insar.stage13_cumulative_relationship import reconstruct_stage13
from groundwater_insar.stage14_hydromechanical_states import reconstruct_stage14

cfg = load_config()

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("Repository  :", cfg["paths"]["repository_root"])
print("Regenerated :", cfg["paths"]["regenerated_root"])

## 1. Provenance

The Git branch, commit and working-tree state are recorded for reproducibility.

In [ ]:
repo = Path(cfg["paths"]["repository_root"])

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(repo), *args],
        text=True
    ).strip()

print("Git branch :", git("branch", "--show-current"))
print("Git commit :", git("rev-parse", "HEAD"))

status = git("status", "--porcelain")

print("Git dirty  :", bool(status))

if status:
    print("\nUncommitted files:")
    print(status)

## 2. Stage 11 — SBAS stack-boundary QC

The temporal relationship between consecutive SBAS stacks is inspected before stitching.

Boundary classes are:

- `EXACT_COMMON_DATE`
- `SHORT_GAP`
- `LONG_GAP`
- `TEMPORAL_OVERLAP_NO_EXACT_DATE`

The `1395 → 1396` boundary contains a long 84-day gap.  
The continuous stitched analysis therefore begins with **stack 1396**.

In [ ]:
stage11, stage11_out = reconstruct_stage11(cfg)

print("Boundaries:", len(stage11))

display(stage11)

print("\nStatus counts:")
print(
    stage11["Status"]
    .value_counts()
    .to_string()
)

## 3. Stage 12 — Cumulative SBAS stack stitching

Stacks `1396–1403` are stitched sequentially for the 37 eligible piezometers.

Historical stitching rules:

- **1396**: base stack;
- **short positive gaps**: zero-bridge between the previous final epoch and the next first epoch;
- **exact common dates**: median offset at common valid dates;
- **temporal overlap without an exact common date**: interpolation at the temporal midpoint.

The resulting quantity remains cumulative vertical subsidence in millimetres.

In [ ]:
(
    stage12_epochs,
    stage12_qc,
    stage12_epochs_out,
    stage12_qc_out,
) = reconstruct_stage12(cfg)

print("Stitched epoch rows :", len(stage12_epochs))
print("Wells               :", stage12_epochs["Piezometer"].nunique())
print("Boundary QC rows    :", len(stage12_qc))

print("\nBoundary methods:")
print(
    stage12_qc["Method"]
    .value_counts()
    .to_string()
)

display(stage12_qc.head(10))

In [ ]:
offset_summary = (
    stage12_qc
    .groupby(
        ["FromStack", "ToStack", "Method"]
    )["Applied_Offset_mm"]
    .agg(["count", "min", "median", "max"])
)

display(offset_summary)

## 4. Stage 13 — Cumulative groundwater–InSAR analysis

The stitched epoch series is aggregated to a monthly cumulative series.

For every piezometer:

- monthly stitched cumulative subsidence is calculated;
- the preferred SBAS stack is selected;
- monthly groundwater head is joined;
- groundwater drawdown is referenced to `1395/12`;
- cumulative InSAR displacement is referenced to the same baseline month;
- cumulative groundwater–subsidence relationships are calculated.

Positive cumulative drawdown represents groundwater-level decline.

In [ ]:
(
    stage13_monthly,
    stage13_metrics,
    stage13_monthly_out,
    stage13_metrics_out,
) = reconstruct_stage13(cfg)

print("Monthly rows :", len(stage13_monthly))
print("Wells        :", stage13_monthly["Piezometer"].nunique())
print(
    "Valid pairs  :",
    int(stage13_monthly["Valid_Cumulative_Pair"].sum())
)

print("\nMetric N distribution:")
print(
    stage13_metrics["N"]
    .value_counts()
    .sort_index()
    .to_string()
)

display(stage13_metrics.head(10))

In [ ]:
example = (
    stage13_monthly[
        stage13_monthly["Piezometer"] == "P01"
    ]
    .sort_values("MonthIndex")
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    example["MonthIndex"],
    example["Cumulative_Subsidence_from_Base_mm"],
    marker="o",
    markersize=3,
)

ax.set_xlabel("MonthIndex")
ax.set_ylabel("Cumulative subsidence from baseline (mm)")
ax.set_title("Example stitched cumulative InSAR series — P01")

plt.tight_layout()
plt.show()

## 5. Stage 14 — Monthly hydro-mechanical states

Monthly groundwater and subsidence increments are calculated from the cumulative Stage 13 series.

A transition is `Primary_Valid` only when:

- months are exactly consecutive;
- both observations belong to the same stitched SBAS stack;
- groundwater and InSAR increments are both available.

Cross-stack transitions are deliberately excluded from the primary state analysis.

Four sign-based states are then defined:

- `DECLINE_AND_SUBSIDENCE`
- `DECLINE_BUT_UPLIFT`
- `RECOVERY_BUT_SUBSIDENCE`
- `RECOVERY_AND_UPLIFT`

All invalid, cross-stack or zero-sign cases remain `OTHER_OR_INVALID`.

In [ ]:
(
    stage14_monthly,
    stage14_summary,
    stage14_monthly_out,
    stage14_summary_out,
) = reconstruct_stage14(cfg)

print("Monthly rows      :", len(stage14_monthly))
print(
    "Primary valid     :",
    int(stage14_monthly["Primary_Valid"].sum())
)
print(
    "Cross-stack rows  :",
    int(stage14_monthly["Cross_Stack"].sum())
)

print("\nState counts:")
print(
    stage14_monthly["HydroMechanical_State"]
    .value_counts()
    .to_string()
)

display(stage14_summary.head(10))

In [ ]:
state_counts = (
    stage14_monthly["HydroMechanical_State"]
    .value_counts()
)

ax = state_counts.plot(
    kind="bar",
    figsize=(9, 5),
)

ax.set_xlabel("Hydro-mechanical state")
ax.set_ylabel("Number of monthly records")
ax.set_title("Stage 14 hydro-mechanical state counts")

plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
recovery_summary = pd.DataFrame({
    "Metric": [
        "Eligible wells",
        "Primary valid transitions",
        "Recovery + continued subsidence records",
        "Decline + subsidence records",
    ],
    "Value": [
        stage14_monthly["Piezometer"].nunique(),
        int(stage14_monthly["Primary_Valid"].sum()),
        int(
            (
                stage14_monthly["HydroMechanical_State"]
                == "RECOVERY_BUT_SUBSIDENCE"
            ).sum()
        ),
        int(
            (
                stage14_monthly["HydroMechanical_State"]
                == "DECLINE_AND_SUBSIDENCE"
            ).sum()
        ),
    ],
})

display(recovery_summary)

## 6. Dependency checkpoint

Stage 14 produces the canonical `Primary_Valid` monthly transition panel.

This panel contains **3033 valid monthly transitions** and is the correct input support for the later:

- Stage 15 extended CCF analysis;
- Stage 16 distributed-lag analysis.

This dependency is important because the earlier Stage 15 draft based directly on Stage 07 produced different sample sizes and correlations.

In [ ]:
panel = stage14_monthly[
    stage14_monthly["Primary_Valid"]
].copy()

print("Stage-15/16 source-panel rows :", len(panel))
print("Wells                         :", panel["Piezometer"].nunique())

print("\nRows per well:")
print(
    panel.groupby("Piezometer")
    .size()
    .value_counts()
    .sort_index()
    .to_string()
)

## 7. Frozen-baseline regression

Stages 11–14 are compared against the frozen terminal baseline.

No stage is accepted unless the historical output is reproduced within the established regression tolerances.

In [ ]:
regenerated_root = Path(
    cfg["paths"]["regenerated_root"]
)

regression_runs = [
    (
        11,
        regenerated_root / "stage11",
        "11_Stage11_Stitching_QC_Regression.csv",
    ),
    (
        12,
        regenerated_root / "stage12",
        "12_Stage12_Stitched_Cumulative_Regression.csv",
    ),
    (
        13,
        regenerated_root / "stage13",
        "13_Stage13_Cumulative_GW_InSAR_Regression.csv",
    ),
    (
        14,
        regenerated_root / "stage14",
        "14_Stage14_HydroMechanical_Regression.csv",
    ),
]

for stage, candidate_dir, report_name in regression_runs:

    print("\n" + "=" * 90)
    print(f"REGRESSION — STAGE {stage:02d}")
    print("=" * 90)

    cmd = [
        sys.executable,
        "-m",
        "groundwater_insar.regression",
        "--candidate-dir",
        str(candidate_dir),
        "--stage",
        f"{stage:02d}",
        "--report-name",
        report_name,
    ]

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Stage {stage:02d} regression failed."
        )

print("\nSTAGES 11–14 REGRESSION: PASS")

## 8. Scientific checkpoint

The stitched analysis establishes a continuous post-1396 cumulative InSAR framework for the 37 eligible piezometers.

The monthly hydro-mechanical classification shows that groundwater recovery and continued subsidence can occur simultaneously at many monthly transitions.

This observation is descriptive and does not by itself establish a unique lag time or a specific compaction mechanism. Later stages evaluate persistence and post-recovery behaviour more explicitly.

In [ ]:
summary = pd.DataFrame({
    "Stage": [
        "11 — Stack-boundary QC",
        "12 — Cumulative stitching",
        "13 — Cumulative GW–InSAR",
        "14 — Hydro-mechanical states",
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
    ],
})

display(summary)

print()
print(
    "04_STITCHING_CUMULATIVE_HYDROMECHANICAL NOTEBOOK: PASS"
)